# PySpark AML Transaction Monitoring

End-to-end Anti-Money Laundering post-transaction analysis pipeline.

**Stack:** PySpark 3.x, Python 3.10+ | *Author: Ridhan Parvendhan | github.com/RidhanPar*

In [1]:
!pip install pyspark --quiet

In [2]:
import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, BooleanType

spark = (
    SparkSession.builder
    .appName('AML_Transaction_Monitoring')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 4.0.3


## Synthetic Data Generation

> All data is entirely synthetic for demonstration purposes only.

In [3]:
random.seed(42)

N_TRANSACTIONS = 10_000
N_CUSTOMERS    = 500
N_ACCOUNTS     = 600
START_DATE     = datetime(2024, 1, 1)

CURRENCIES          = ['EUR', 'USD', 'GBP', 'SEK', 'CHF']
CHANNELS            = ['SEPA', 'SWIFT', 'FASTER_PAYMENTS', 'INTERNAL', 'CARD']
HIGH_RISK_COUNTRIES = {'CY', 'MT', 'PAN', 'BVI', 'SCH'}
COUNTRY_POOL        = ['LV', 'EE', 'LT', 'FI', 'SE', 'DE', 'GB', 'CY', 'MT', 'PAN', 'BVI']

customer_ids = [f'CUST{str(i).zfill(5)}' for i in range(N_CUSTOMERS)]
account_ids  = [f'ACC{str(i).zfill(6)}'  for i in range(N_ACCOUNTS)]

def random_amount(suspicious=False):
    if suspicious:
        return round(random.uniform(9_000, 9_999), 2)
    return round(random.expovariate(1 / 2500), 2)

def make_transaction(txn_id):
    s  = random.random() < 0.08
    ts = START_DATE + timedelta(
        days=random.randint(0, 364), hours=random.randint(0, 23),
        minutes=random.randint(0, 59), seconds=random.randint(0, 59)
    )
    return {
        'transaction_id':      f'TXN{str(txn_id).zfill(8)}',
        'customer_id':         random.choice(customer_ids),
        'originator_account':  random.choice(account_ids),
        'beneficiary_account': random.choice(account_ids),
        'amount':              random_amount(s),
        'currency':            random.choice(CURRENCIES),
        'channel':             random.choice(CHANNELS),
        'originator_country':  random.choice(COUNTRY_POOL),
        'beneficiary_country': random.choice(COUNTRY_POOL) if s else random.choice(['LV','EE','LT','FI','SE','DE','GB']),
        'timestamp':           ts,
        'is_flagged_source':   s,
    }

records = [make_transaction(i) for i in range(N_TRANSACTIONS)]
print(f'Generated {len(records):,} synthetic transactions.')

Generated 10,000 synthetic transactions.


## Schema and DataFrame Creation

In [4]:
SCHEMA = StructType([
    StructField('transaction_id',      StringType(),    nullable=False),
    StructField('customer_id',         StringType(),    nullable=False),
    StructField('originator_account',  StringType(),    nullable=True),
    StructField('beneficiary_account', StringType(),    nullable=True),
    StructField('amount',              DoubleType(),    nullable=False),
    StructField('currency',            StringType(),    nullable=False),
    StructField('channel',             StringType(),    nullable=False),
    StructField('originator_country',  StringType(),    nullable=True),
    StructField('beneficiary_country', StringType(),    nullable=True),
    StructField('timestamp',           TimestampType(), nullable=False),
    StructField('is_flagged_source',   BooleanType(),   nullable=False),
])

txn_df = spark.createDataFrame(records, schema=SCHEMA)
txn_df.cache()
txn_df.printSchema()
print(f'Row count: {txn_df.count():,}')

root
 |-- transaction_id: string (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- originator_account: string (nullable = true)
 |-- beneficiary_account: string (nullable = true)
 |-- amount: double (nullable = false)
 |-- currency: string (nullable = false)
 |-- channel: string (nullable = false)
 |-- originator_country: string (nullable = true)
 |-- beneficiary_country: string (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- is_flagged_source: boolean (nullable = false)

Row count: 10,000


## Data Quality Validation

In [5]:
critical_cols = ['transaction_id', 'customer_id', 'amount', 'currency', 'timestamp']
txn_df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in critical_cols]).show()
print('Invalid amounts:', txn_df.filter(F.col('amount') <= 0).count())
print('Duplicates:', txn_df.count() - txn_df.select('transaction_id').distinct().count())
txn_df.groupBy('currency').count().orderBy(F.desc('count')).show()
txn_df.groupBy('channel').count().orderBy(F.desc('count')).show()

+--------------+-----------+------+--------+---------+
|transaction_id|customer_id|amount|currency|timestamp|
+--------------+-----------+------+--------+---------+
|             0|          0|     0|       0|        0|
+--------------+-----------+------+--------+---------+

Invalid amounts: 0
Duplicates: 0
+--------+-----+
|currency|count|
+--------+-----+
|     SEK| 2073|
|     EUR| 1993|
|     USD| 1986|
|     GBP| 1984|
|     CHF| 1964|
+--------+-----+

+---------------+-----+
|        channel|count|
+---------------+-----+
|       INTERNAL| 2018|
|           SEPA| 2001|
|          SWIFT| 1998|
|FASTER_PAYMENTS| 1992|
|           CARD| 1991|
+---------------+-----+



## Feature Engineering with Window Functions

Rolling aggregates per customer reveal unusual activity relative to recent behaviour.

In [6]:
txn_enriched = (
    txn_df
    .withColumn('date',        F.to_date('timestamp'))
    .withColumn('hour',        F.hour('timestamp'))
    .withColumn('day_of_week', F.dayofweek('timestamp'))
    .withColumn('is_weekend',  F.col('day_of_week').isin(1, 7))
    .withColumn('is_offhours', (F.col('hour') < 6) | (F.col('hour') >= 22))
    .withColumn('involves_high_risk_country',
        F.col('originator_country').isin(list(HIGH_RISK_COUNTRIES)) |
        F.col('beneficiary_country').isin(list(HIGH_RISK_COUNTRIES)))
)

W_7D    = Window.partitionBy('customer_id').orderBy(F.unix_timestamp('timestamp')).rangeBetween(-7*86400, 0)
W_CUMUL = Window.partitionBy('customer_id').orderBy(F.unix_timestamp('timestamp')).rowsBetween(Window.unboundedPreceding, 0)
W_LAG   = Window.partitionBy('customer_id').orderBy('timestamp')

txn_features = (
    txn_enriched
    .withColumn('rolling_7d_amount',       F.sum('amount').over(W_7D))
    .withColumn('rolling_7d_count',        F.count('transaction_id').over(W_7D))
    .withColumn('rolling_7d_avg',          F.avg('amount').over(W_7D))
    .withColumn('cumulative_amount',       F.sum('amount').over(W_CUMUL))
    .withColumn('cumulative_count',        F.count('transaction_id').over(W_CUMUL))
    .withColumn('prev_ts',                 F.lag('timestamp', 1).over(W_LAG))
    .withColumn('seconds_since_last_txn',  F.unix_timestamp('timestamp') - F.unix_timestamp('prev_ts'))
    .withColumn('amount_vs_7d_avg_ratio',
        F.when(F.col('rolling_7d_avg') > 0, F.col('amount') / F.col('rolling_7d_avg')).otherwise(None))
    .drop('prev_ts')
)
txn_features.cache()
print('Feature engineering complete.')
txn_features.select('transaction_id','customer_id','amount','rolling_7d_amount','rolling_7d_count','rolling_7d_avg','seconds_since_last_txn').show(5, truncate=False)

Feature engineering complete.
+--------------+-----------+-------+------------------+----------------+------------------+----------------------+
|transaction_id|customer_id|amount |rolling_7d_amount |rolling_7d_count|rolling_7d_avg    |seconds_since_last_txn|
+--------------+-----------+-------+------------------+----------------+------------------+----------------------+
|TXN00009842   |CUST00005  |2613.28|2613.28           |1               |2613.28           |NULL                  |
|TXN00000938   |CUST00005  |2005.68|2005.68           |1               |2005.68           |878672                |
|TXN00001872   |CUST00005  |814.8  |814.8             |1               |814.8             |1129717               |
|TXN00007063   |CUST00005  |2603.34|3418.1400000000003|2               |1709.0700000000002|208871                |
|TXN00007768   |CUST00005  |1993.56|1993.56           |1               |1993.56           |3163992               |
+--------------+-----------+-------+--------------

## AML Typology Detection

| Typology | Rule |
|---|---|
| Structuring | Amount 9,000-9,999 AND 3+ txns in 7 days |
| High-velocity layering | 5+ txns and 20k+ volume in 7 days |
| High-risk country routing | Originator/beneficiary in weak-AML jurisdiction |
| Amount spike | Current amount >= 3x customer 7-day average |
| Rapid succession | Less than 5 minutes since previous transaction |

In [7]:
txn_flagged = (
    txn_features
    .withColumn('flag_structuring',
        F.col('amount').between(9_000, 9_999.99) & (F.col('rolling_7d_count') >= 3))
    .withColumn('flag_high_velocity',
        (F.col('rolling_7d_count') >= 5) & (F.col('rolling_7d_amount') >= 20_000))
    .withColumn('flag_high_risk_country', F.col('involves_high_risk_country'))
    .withColumn('flag_amount_spike',
        (F.col('amount_vs_7d_avg_ratio') >= 3.0) & F.col('amount_vs_7d_avg_ratio').isNotNull())
    .withColumn('flag_rapid_succession',
        (F.col('seconds_since_last_txn') <= 300) & F.col('seconds_since_last_txn').isNotNull())
    .withColumn('flag_count',
        F.col('flag_structuring').cast('int') + F.col('flag_high_velocity').cast('int') +
        F.col('flag_high_risk_country').cast('int') + F.col('flag_amount_spike').cast('int') +
        F.col('flag_rapid_succession').cast('int'))
)
txn_flagged.cache()

flag_cols = ['flag_structuring','flag_high_velocity','flag_high_risk_country','flag_amount_spike','flag_rapid_succession']
txn_flagged.select([F.sum(F.col(c).cast('int')).alias(c) for c in flag_cols]).show()

+----------------+------------------+----------------------+-----------------+---------------------+
|flag_structuring|flag_high_velocity|flag_high_risk_country|flag_amount_spike|flag_rapid_succession|
+----------------+------------------+----------------------+-----------------+---------------------+
|              55|                 1|                  3879|                0|                    2|
+----------------+------------------+----------------------+-----------------+---------------------+



## Risk Scoring and Alert Triage

In [8]:
WEIGHTS = {
    'flag_structuring':       30,
    'flag_high_velocity':     25,
    'flag_high_risk_country': 20,
    'flag_amount_spike':      15,
    'flag_rapid_succession':  10,
}
risk_score_expr = sum(F.col(f).cast('int') * w for f, w in WEIGHTS.items())

txn_scored = (
    txn_flagged
    .withColumn('risk_score', risk_score_expr)
    .withColumn('risk_tier',
        F.when(F.col('risk_score') >= 50, 'HIGH')
         .when(F.col('risk_score') >= 25, 'MEDIUM')
         .when(F.col('risk_score') >   0, 'LOW')
         .otherwise('NONE'))
)
txn_scored.cache()
txn_scored.groupBy('risk_tier').count().orderBy(F.desc('count')).show()
txn_scored.filter(F.col('risk_tier') == 'HIGH')\
    .select('transaction_id','customer_id','amount','risk_score','risk_tier',
            'flag_structuring','flag_high_velocity','flag_high_risk_country')\
    .orderBy(F.desc('risk_score')).show(10, truncate=False)

+---------+-----+
|risk_tier|count|
+---------+-----+
|     NONE| 6090|
|      LOW| 3855|
|   MEDIUM|   29|
|     HIGH|   26|
+---------+-----+

+--------------+-----------+-------+----------+---------+----------------+------------------+----------------------+
|transaction_id|customer_id|amount |risk_score|risk_tier|flag_structuring|flag_high_velocity|flag_high_risk_country|
+--------------+-----------+-------+----------+---------+----------------+------------------+----------------------+
|TXN00007592   |CUST00071  |9837.53|75        |HIGH     |true            |true              |true                  |
|TXN00002782   |CUST00008  |9702.47|50        |HIGH     |true            |false             |true                  |
|TXN00004789   |CUST00318  |9138.3 |50        |HIGH     |true            |false             |true                  |
|TXN00009924   |CUST00052  |9073.27|50        |HIGH     |true            |false             |true                  |
|TXN00008444   |CUST00143  |9231.81|

## Customer-Level Risk Aggregation

In [9]:
customer_risk = (
    txn_scored
    .groupBy('customer_id')
    .agg(
        F.count('transaction_id')                          .alias('total_transactions'),
        F.sum('amount')                                    .alias('total_volume_eur'),
        F.avg('amount')                                    .alias('avg_transaction_amount'),
        F.max('amount')                                    .alias('max_transaction_amount'),
        F.sum(F.col('risk_score'))                         .alias('cumulative_risk_score'),
        F.max('risk_score')                                .alias('peak_risk_score'),
        F.sum(F.col('flag_structuring').cast('int'))       .alias('structuring_flags'),
        F.sum(F.col('flag_high_velocity').cast('int'))     .alias('velocity_flags'),
        F.sum(F.col('flag_high_risk_country').cast('int')) .alias('high_risk_country_flags'),
        F.countDistinct('beneficiary_country')             .alias('distinct_beneficiary_countries'),
        F.countDistinct('beneficiary_account')             .alias('distinct_beneficiary_accounts'),
    )
    .withColumn('customer_risk_tier',
        F.when(F.col('peak_risk_score') >= 50, 'HIGH')
         .when(F.col('peak_risk_score') >= 25, 'MEDIUM')
         .when(F.col('peak_risk_score') >   0, 'LOW')
         .otherwise('NONE'))
    .orderBy(F.desc('cumulative_risk_score'))
)
customer_risk.cache()
customer_risk.show(15, truncate=False)

+-----------+------------------+-----------------+----------------------+----------------------+---------------------+---------------+-----------------+--------------+-----------------------+------------------------------+-----------------------------+------------------+
|customer_id|total_transactions|total_volume_eur |avg_transaction_amount|max_transaction_amount|cumulative_risk_score|peak_risk_score|structuring_flags|velocity_flags|high_risk_country_flags|distinct_beneficiary_countries|distinct_beneficiary_accounts|customer_risk_tier|
+-----------+------------------+-----------------+----------------------+----------------------+---------------------+---------------+-----------------+--------------+-----------------------+------------------------------+-----------------------------+------------------+
|CUST00305  |32                |84989.82         |2655.931875           |9663.8                |360                  |20             |0                |0             |18               

## Alert Queue Generation

In [10]:
ALERT_THRESHOLD = 25

alert_queue = (
    txn_scored
    .filter(F.col('risk_score') >= ALERT_THRESHOLD)
    .select('transaction_id','customer_id','timestamp','amount','currency','channel',
            'originator_country','beneficiary_country','risk_score','risk_tier',
            'flag_count','rolling_7d_amount','rolling_7d_count')
    .orderBy(F.desc('risk_score'), F.desc('amount'))
)
n = alert_queue.count()
print(f'Alerts: {n:,} / {txn_scored.count():,} ({n/txn_scored.count():.1%} alert rate)')
alert_queue.show(20, truncate=False)

Alerts: 55 / 10,000 (0.5% alert rate)
+--------------+-----------+-------------------+-------+--------+---------------+------------------+-------------------+----------+---------+----------+------------------+----------------+
|transaction_id|customer_id|timestamp          |amount |currency|channel        |originator_country|beneficiary_country|risk_score|risk_tier|flag_count|rolling_7d_amount |rolling_7d_count|
+--------------+-----------+-------------------+-------+--------+---------------+------------------+-------------------+----------+---------+----------+------------------+----------------+
|TXN00007592   |CUST00071  |2024-07-19 23:55:58|9837.53|EUR     |SEPA           |MT                |SE                 |75        |HIGH     |3         |42591.43          |5               |
|TXN00002577   |CUST00336  |2024-12-20 04:07:30|9973.65|USD     |FASTER_PAYMENTS|LT                |BVI                |50        |HIGH     |2         |13652.06          |3               |
|TXN00002416   |C

## Precision / Recall Evaluation

In [11]:
from pyspark.sql.functions import col

eval_df = txn_scored.withColumn('pred_pos', F.col('risk_score') >= ALERT_THRESHOLD)
tp = eval_df.filter( col('is_flagged_source') &  col('pred_pos')).count()
fp = eval_df.filter(~col('is_flagged_source') &  col('pred_pos')).count()
tn = eval_df.filter(~col('is_flagged_source') & ~col('pred_pos')).count()
fn = eval_df.filter( col('is_flagged_source') & ~col('pred_pos')).count()
prec = tp/(tp+fp) if tp+fp>0 else 0
rec  = tp/(tp+fn) if tp+fn>0 else 0
f1   = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
print(f'TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}')

TP=47  FP=8  TN=9184  FN=761
Precision=0.855  Recall=0.058  F1=0.109


## Monthly Trend Reporting

In [12]:
(
    txn_scored
    .withColumn('year',  F.year('timestamp'))
    .withColumn('month', F.month('timestamp'))
    .groupBy('year', 'month')
    .agg(
        F.count('transaction_id').alias('total_transactions'),
        F.sum('amount').alias('total_volume_eur'),
        F.sum((F.col('risk_score') >= ALERT_THRESHOLD).cast('int')).alias('alerts_raised'),
        F.sum(F.col('flag_structuring').cast('int')).alias('structuring_flags'),
    )
    .withColumn('alert_rate_pct', F.round(F.col('alerts_raised') / F.col('total_transactions') * 100, 2))
    .orderBy('year', 'month')
    .show(12, truncate=False)
)

+----+-----+------------------+------------------+-------------+-----------------+--------------+
|year|month|total_transactions|total_volume_eur  |alerts_raised|structuring_flags|alert_rate_pct|
+----+-----+------------------+------------------+-------------+-----------------+--------------+
|2024|1    |823               |2477201.83        |0            |0                |0.0           |
|2024|2    |808               |2365285.85        |5            |5                |0.62          |
|2024|3    |838               |2468625.5199999996|1            |1                |0.12          |
|2024|4    |776               |2280525.6         |6            |6                |0.77          |
|2024|5    |880               |2667482.39        |6            |6                |0.68          |
|2024|6    |815               |2654310.6100000003|6            |6                |0.74          |
|2024|7    |884               |2792658.4899999998|10           |10               |1.13          |
|2024|8    |890     

## Export to Parquet

In [13]:
import os
os.makedirs('./output', exist_ok=True)
alert_queue.coalesce(1).write.mode('overwrite').parquet('./output/alert_queue')
customer_risk.coalesce(1).write.mode('overwrite').parquet('./output/customer_risk_profiles')
print('Outputs saved to ./output/')

Outputs saved to ./output/


In [14]:
for df in [txn_df, txn_features, txn_flagged, txn_scored, customer_risk]:
    df.unpersist()
spark.stop()
print('Done.')

Done.


---
## Summary

| Step | Technique |
|---|---|
| Data generation | 10k synthetic transactions with AML patterns |
| Schema validation | Explicit StructType + null/duplicate checks |
| Feature engineering | Window functions: rolling 7d sum/avg/count, lag, cumulative |
| Typology detection | 5 rule-based flags: structuring, velocity, high-risk country, spike, rapid succession |
| Risk scoring | Weighted composite score + HIGH/MEDIUM/LOW tiering |
| Customer aggregation | GroupBy + multi-agg for case-level view |
| Evaluation | Precision/Recall/F1 vs synthetic ground truth |
| Trend reporting | Month-over-month alert rate for regulatory MI |
| Export | Parquet for downstream systems |

**Disclaimer:** all data is synthetically generated.